# Synthetic Minority Oversampling Technique (SMOTE) - Starter Notebook
This notebook implements a simple SMOTE routine from scratch to generate synthetic minority examples and rebalance an imbalanced dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors

In [ ]:
X, y = make_classification(
    n_samples=400,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    weights=[0.9, 0.1],
    class_sep=1.2,
    random_state=42,
)

In [ ]:
def simple_smote(minority_samples, samples_to_create, k_neighbors=5, random_state=42):
    rng = np.random.default_rng(random_state)
    model = NearestNeighbors(n_neighbors=min(k_neighbors, len(minority_samples)))
    model.fit(minority_samples)
    neighbors = model.kneighbors(minority_samples, return_distance=False)

    synthetic = []
    for _ in range(samples_to_create):
        sample_index = rng.integers(len(minority_samples))
        sample = minority_samples[sample_index]
        candidate_neighbors = neighbors[sample_index][1:] if len(neighbors[sample_index]) > 1 else neighbors[sample_index]
        neighbor_index = rng.choice(candidate_neighbors)
        neighbor = minority_samples[neighbor_index]
        gap = rng.random()
        synthetic.append(sample + gap * (neighbor - sample))
    return np.array(synthetic)


minority = X[y == 1]
majority = X[y == 0]
synthetic_samples = simple_smote(minority, len(majority) - len(minority))
X_resampled = np.vstack([X, synthetic_samples])
y_resampled = np.concatenate([y, np.ones(len(synthetic_samples), dtype=int)])
pd.Series(y_resampled).value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.6, label='Majority')
axes[0].scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.8, label='Minority')
axes[0].set_title('Original Imbalanced Dataset')
axes[0].legend()

axes[1].scatter(X_resampled[y_resampled == 0, 0], X_resampled[y_resampled == 0, 1], alpha=0.5, label='Majority')
axes[1].scatter(X_resampled[y_resampled == 1, 0], X_resampled[y_resampled == 1, 1], alpha=0.6, label='Minority + Synthetic')
axes[1].set_title('Dataset After Simple SMOTE')
axes[1].legend()
plt.tight_layout()
plt.show()